In [1]:
import numpy as np
import torch
import clip
from transformers import AutoImageProcessor, AutoModel
from qdrant_client import QdrantClient, models
import pandas as pd
from PIL import Image
import os
import time

os.getpid()

/home/inna/.cache/pypoetry/virtualenvs/sneakersearch-z_mDpiHd-py3.12/lib/python3.12/site-packages/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
/home/inna/.cache/pypoetry/virtualenvs/sneakersearch-z_mDpiHd-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


15323

In [2]:
#поднимаем ВБД с маппингом в корень прокта
# в корне проекта выполнить 

#docker run -p 6333:6333 -p 6334:6334 -v "$(pwd)/qdrant_storage:/qdrant/storage" qdrant/qdrant

# инициализация клиента ВБД
client = QdrantClient("http://localhost:6333")

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
# client.recreate_collection(
#         collection_name="sneakers",
#         vectors_config=models.VectorParams(size=512, distance=models.Distance.COSINE),
#     )

/tmp/ipykernel_15972/1366355860.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [3]:
prefix_path = "/home/inna/Рабочий стол/SneakerSearch/data/"

lamoda_data = pd.read_csv(prefix_path+"lamoda_data.csv", sep=";")
lamoda_data.head()

,brand,model,category,color,description,lamoda_photo,title_photo,path_to_lamoda_photo
0,Thomas Munz,Thomas Munz Кеды Полнота D (4),Низкие кеды,розовый,NaN,https://a.lmcdn.ru/product/M/P/MP002XW1BIY9_26...,lamoda_photos/Thomas_Munz_Thomas_Munz_Кеды_Пол...,//a.lmcdn.ru/product/M/P/MP002XW1BIY9_26991175...
1,Thomas Munz,Thomas Munz Кеды Полнота D (4),Низкие кеды,розовый,NaN,https://a.lmcdn.ru/product/M/P/MP002XW1BIY9_26...,lamoda_photos/Thomas_Munz_Thomas_Munz_Кеды_Пол...,//a.lmcdn.ru/product/M/P/MP002XW1BIY9_26991175...
2,Thomas Munz,Thomas Munz Кеды Полнота D (4),Низкие кеды,розовый,NaN,https://a.lmcdn.ru/product/M/P/MP002XW1BIY9_26...,lamoda_photos/Thomas_Munz_Thomas_Munz_Кеды_Пол...,//a.lmcdn.ru/product/M/P/MP002XW1BIY9_26991175...
3,Thomas Munz,Thomas Munz Кеды Полнота D (4),Низкие кеды,розовый,NaN,https://a.lmcdn.ru/product/M/P/MP002XW1BIY9_26...,lamoda_photos/Thomas_Munz_Thomas_Munz_Кеды_Пол...,//a.lmcdn.ru/product/M/P/MP002XW1BIY9_26991175...
4,Thomas Munz,Thomas Munz Кеды Полнота D (4),Низкие кеды,розовый,NaN,https://a.lmcdn.ru/product/M/P/MP002XW1BIY9_26...,lamoda_photos/Thomas_Munz_Thomas_Munz_Кеды_Пол...,//a.lmcdn.ru/product/M/P/MP002XW1BIY9_26991175...


In [4]:
# оставляем только записи с титульными фото
lamoda_data = lamoda_data.drop_duplicates(subset="path_to_lamoda_photo")
print(len(lamoda_data))

2251


In [5]:
lamoda_data = lamoda_data.drop_duplicates(subset="title_photo")
print(len(lamoda_data))

1011


## CLIP emb

In [6]:
# инизацлизация модели для ембеддингов
model_name = "ViT-B/32" #338M params
model, preprocess = clip.load(model_name, device=device) 

In [7]:
img_path = prefix_path+"lamoda_photos/Under_Armour_Under_Armour_Кроссовки_UA_W_Charged_P_0.jpg"
image = Image.open(img_path).convert("RGB")
preproc_image = preprocess(image).unsqueeze(0).to(device)
with torch.no_grad():
    image_emb = model.encode_image(preproc_image)
image_emb.shape

torch.Size([1, 512])

In [8]:
def get_image_embedding(img_path):
    image = Image.open(img_path).convert('RGB')

    preproc_image = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        image_emb = model.encode_image(preproc_image)
        
    # Нормализуем эмбеддинг (это важно для поиска в Qdrant)
    image_emb /= image_emb.norm(dim=-1, keepdim=True)
    
    return image_emb.cpu().numpy().flatten()

In [9]:
def create_db(
        client: QdrantClient,
        collection_name: str, 
        emb_dim: int,
        data):
    
    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(size=emb_dim, distance=models.Distance.COSINE),
        )

        for idx, row in data.iterrows():
                img_path = row["title_photo"]
                try:
                    image_emb = get_image_embedding(img_path)

                    point = models.PointStruct(
                        id=idx, 
                        vector=image_emb.tolist(), 
                        payload={
                            "brand": row["brand"],
                            "model": row["model"],
                            "color": row["color"],
                            "path_to_photo": os.path.join(prefix_path, row["title_photo"]) # cохраняем путь для отображения!
                        }
                    )

                    client.upsert(
                        collection_name=collection_name, 
                        points=[point]
                        )
                except: 
                    continue
    else:
        print("Коллекция уже существует")

In [ ]:
emb_dim = 512
collection_name = "Sneakers_CLIP"

create_db(
        client,
        collection_name, 
        emb_dim,
        lamoda_data)

## Metrics CLIP

In [10]:
user_data = pd.read_csv(prefix_path+"user_data.csv", sep=";")
user_data = user_data.drop_duplicates()
user_data = user_data.drop_duplicates(subset="path_to_user_photo")
user_data = user_data.sample(frac=1).reset_index(drop=True) #shuffle data
print(len(user_data))

1439


In [7]:
user_data

,brand,model,category,color,description,title_photo,path_to_title_photo,user_photo,path_to_user_photo
0,Fila,Fila Кроссовки TRACE LOW M,Кроссовки,белый,Кроссовки выполнены из текстиля и синтетическо...,//a.lmcdn.ru/product/M/P/MP002XW17JUS_24270651...,lamoda_photos/Fila_Fila_Кроссовки_TRACE_LOW_M_...,https://a.lmcdn.ru/photoreview/?key=847d3f82-3...,user_photos/Fila_Fila_Кроссовки_TRACE_LOW_M_2.jpg
1,Matrix Sport,Matrix Sport Кеды Sneakers,Низкие кеды,черный,NaN,//a.lmcdn.ru/product/M/P/MP002XW1JCJ9_28325988...,lamoda_photos/Matrix_Sport_Matrix_Sport_Кеды_S...,https://a.lmcdn.ru/photoreview/?key=b98ad6bb-d...,user_photos/Matrix_Sport_Matrix_Sport_Кеды_Sne...
2,Enrico Coveri,Enrico Coveri Кроссовки,Низкие кроссовки,черный,Кроссовки выполнены из синтетической кожи. Дет...,//a.lmcdn.ru/product/M/P/MP002XW17O0J_24354920...,lamoda_photos/Enrico_Coveri_Enrico_Coveri_Крос...,https://a.lmcdn.ru/photoreview/?key=d917c7c2-f...,user_photos/Enrico_Coveri_Enrico_Coveri_Кроссо...
3,PUMA,PUMA Кроссовки PUMA CATCH SOLEIL,Низкие кроссовки,белый,Кроссовки PUMA Catch Soleil — стильная и удобн...,//a.lmcdn.ru/product/R/T/RTLAFB521401_32319344...,lamoda_photos/PUMA_PUMA_Кроссовки_PUMA_CATCH_S...,https://a.lmcdn.ru/photoreview/?key=cdb6c743-6...,user_photos/PUMA_PUMA_Кроссовки_PUMA_CATCH_SOL...
4,Nike,Nike Кроссовки Nike Cortez,Низкие кроссовки,голубой,"Кроссовки Nike Cortez - культовая модель, дебю...",//a.lmcdn.ru/product/R/T/RTLAEZ909201_32036323...,lamoda_photos/Nike_Nike_Кроссовки_Nike_Cortez_...,https://a.lmcdn.ru/photoreview/?key=8aff48d9-5...,user_photos/Nike_Nike_Кроссовки_Nike_Cortez_2.jpg
...,...,...,...,...,...,...,...,...,...
1434,PUMA,PUMA Кроссовки Mostro Unlined,Низкие кроссовки,коричневый,Обувь Mostro подходит не всем. Уже более двух ...,//a.lmcdn.ru/product/R/T/RTLAEP878901_29547289...,lamoda_photos/PUMA_PUMA_Кроссовки_Mostro_Unlin...,https://a.lmcdn.ru/photoreview/?key=e7b5d67d-f...,user_photos/PUMA_PUMA_Кроссовки_Mostro_Unlined...
1435,Tommy Jeans,Tommy Jeans Кеды THE GREENWICH FLATFORM,Низкие кеды,белый,NaN,//a.lmcdn.ru/product/R/T/RTLAEV713601_31147001...,lamoda_photos/Tommy_Jeans_Tommy_Jeans_Кеды_THE...,https://a.lmcdn.ru/photoreview/?key=c1430cf6-9...,user_photos/Tommy_Jeans_Tommy_Jeans_Кеды_THE_G...
1436,PUMA,PUMA Кроссовки Speedcat COW,Низкие кроссовки,коричневый,Кроссовки PUMA Speedcat уже несколько десятиле...,//a.lmcdn.ru/product/R/T/RTLAFB525801_32319489...,lamoda_photos/PUMA_PUMA_Кроссовки_Speedcat_COW...,https://a.lmcdn.ru/photoreview/?key=ba23e95b-e...,user_photos/PUMA_PUMA_Кроссовки_Speedcat_COW_2...
1437,Outventure,Outventure Кроссовки London,Низкие кроссовки,синий,Кроссовки с верхом из текстильной сетки. Детал...,//a.lmcdn.ru/product/M/P/MP002XW05YRU_13802714...,lamoda_photos/Outventure_Outventure_Кроссовки_...,https://a.lmcdn.ru/photoreview/?key=43a7b5ec-6...,user_photos/Outventure_Outventure_Кроссовки_Lo...


In [11]:
test_data = user_data[-500:]
image_reference = dict(zip(test_data["path_to_user_photo"], test_data["model"]))

In [12]:
def count_metrics(collection_name):

    recall_5 = 0
    recall_10 = 0
    accuracy = 0
    query_time = []
    N = 0
    unprocessable = []

    for image_path, reference_model in image_reference.items():
        try:
            start = time.perf_counter()
            query = get_image_embedding(prefix_path+image_path)
            candidates = client.query_points(
                                        collection_name=collection_name,
                                        query=query, 
                                        limit=10,           
                                        with_payload=True   # Возвращаем бренд, модель и путь из CSV
                                    ).points
            end = time.perf_counter()
        
            results = [candidate.payload.get("model") for candidate in candidates]
            if reference_model in results:
                recall_10 += 1
            if reference_model in results[:5]:
                recall_5 += 1
            if reference_model == results[0]:
                accuracy += 1
            query_time.append(end-start)
            N +=1

        except:
            unprocessable.append(image_path)
            continue

    print("Unprocessable: ", len(unprocessable))

    return recall_10 / N, recall_5 / N, accuracy / N, np.mean(query_time)

In [13]:
recall_10_clip, recall_5_clip, accuracy_clip, avg_query_time_clip = count_metrics(collection_name)

Unprocessable:  0


In [14]:
df_data = {
    "Accuracy" : [accuracy_clip],
    "Recall@5": [recall_5_clip], 
    "Recall@10": [recall_10_clip],
    "Avg_query_time": [avg_query_time_clip]
}
  
table = pd.DataFrame(data=df_data)
table

,Accuracy,Recall@5,Recall@10,Avg_query_time
0,0.104,0.202,0.25,0.037693


## DINO emb

In [15]:
model_name = 'facebook/dinov2-large' #300M params


processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, device_map="auto")

Loading weights: 100%|██████████| 439/439 [00:00<00:00, 2281.96it/s]


In [16]:
img_path = "/home/inna/Рабочий стол/SneakerSearch/scripts/lamoda_photos/Under_Armour_Under_Armour_Кроссовки_UA_W_Charged_P_0.jpg"
image = Image.open(img_path).convert("RGB")


inputs = processor(images=image, return_tensors="pt").to(device)
with torch.inference_mode():
    outputs = model(**inputs)
    image_emb = outputs.last_hidden_state[:, 0, :] # CLS-token

image_emb.shape

torch.Size([1, 1024])

In [17]:
def get_image_embedding(img_path):
    image = Image.open(img_path).convert('RGB')
    
    preproc_image = processor(images=image, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = model(**preproc_image)
        image_emb = outputs.last_hidden_state[:, 0, :] # CLS-token
        # Нормализуем эмбеддинг (это важно для поиска в Qdrant)
        image_emb /= image_emb.norm(dim=-1, keepdim=True)
    
    return image_emb.cpu().numpy().flatten()

In [18]:
emb_dim = 1024
collection_name = "Sneakers_DINO"

create_db(
        client,
        collection_name, 
        emb_dim,
        lamoda_data)

In [19]:
recall_10_dino, recall_5_dino, accuracy_dino, avg_query_time_dino = count_metrics(collection_name)

Unprocessable:  0


In [21]:
df_data = {
    "Accuracy" : [accuracy_clip, accuracy_dino],
    "Recall@5": [recall_5_clip, recall_5_dino], 
    "Recall@10": [recall_10_clip, recall_10_dino],
    "Avg_query_time": [avg_query_time_clip, avg_query_time_dino]
}
  
table = pd.DataFrame(data=df_data)
table

,Accuracy,Recall@5,Recall@10,Avg_query_time
0,0.104,0.202,0.250,0.037693
1,0.026,0.056,0.072,0.178348
